# Part 1 — XOR and MLPs: neural nets from first principles

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbecken/f5-tests/blob/claude/transformers/notebooks/01_xor_and_mlps.ipynb)

Companion notebook to [Part 1 of the guide](https://github.com/lbecken/f5-tests/blob/claude/transformers/part-1-neural-networks.md). Three programs: XOR with backprop **by hand** in numpy, the same net in PyTorch, and the digit classifier with every part named.

In [ ]:
# Colab already ships torch, sklearn, matplotlib. Locally: pip install torch scikit-learn matplotlib
import os
os.makedirs('../diagrams', exist_ok=True)  # plots land here

## `01_xor_numpy.py`

```
Part 1 - XOR with backpropagation BY HAND (numpy only).

The smallest network that can solve XOR: 2 inputs -> 2 hidden neurons -> 1
output. Nine parameters. Every gradient is written out explicitly so that
PyTorch's loss.backward() never feels like magic again.

Run:  python 01_xor_numpy.py
```

In [ ]:
import numpy as np

# Seed 0 converges. Try seed 1: training plateaus at loss ~0.125, a local
# minimum of this tiny loss landscape -- that plateau IS exercise 1.2.
rng = np.random.default_rng(0)

In [ ]:
# ---------------------------------------------------------------- the data
# The entire "dataset": the truth table of XOR. Shapes: X (4, 2), y (4, 1).
X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = np.array([[0.], [1.], [1.], [0.]])

In [ ]:
# ---------------------------------------------------- the 9 parameters (theta)
# Layer 1: 2 -> 2         W1 (2, 2), b1 (2,)
# Layer 2: 2 -> 1         W2 (1, 2), b2 (1,)
W1 = rng.normal(0, 1.0, size=(2, 2))
b1 = np.zeros(2)
W2 = rng.normal(0, 1.0, size=(1, 2))
b2 = np.zeros(1)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


lr = 0.5          # the learning rate eta
for step in range(10_000):
    # ------------------------------------------------------- FORWARD PASS
    # Each line is one stage of the pipeline  x -> z1 -> h -> z2 -> p -> L
    z1 = X @ W1.T + b1          # (4, 2)  pre-activations, hidden layer
    h = np.tanh(z1)             # (4, 2)  hidden activations
    z2 = h @ W2.T + b2          # (4, 1)  pre-activation, output neuron
    p = sigmoid(z2)             # (4, 1)  prediction in (0, 1)
    loss = np.mean((p - y) ** 2)  # MSE, one number

    # ------------------------------------------------------ BACKWARD PASS
    # Chain rule, applied back-to-front. dX means dLoss/dX, and every dX
    # has the same shape as X. Multiply local derivatives, reuse upstream.
    N = len(X)
    dp = 2.0 * (p - y) / N            # (4, 1)  d/dp of mean((p-y)^2)
    dz2 = dp * p * (1 - p)            # (4, 1)  sigmoid'(z) = p(1-p)
    dW2 = dz2.T @ h                   # (1, 2)  z2 = h W2^T  =>  dW2 = dz2^T h
    db2 = dz2.sum(axis=0)             # (1,)
    dh = dz2 @ W2                     # (4, 2)  gradient flows down to h
    dz1 = dh * (1 - h ** 2)           # (4, 2)  tanh'(z) = 1 - tanh(z)^2
    dW1 = dz1.T @ X                   # (2, 2)
    db1 = dz1.sum(axis=0)             # (2,)

    # ------------------------------------------------------------- UPDATE
    # theta <- theta - eta * dL/dtheta   (gradient descent, section 1.6)
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if step % 1000 == 0:
        print(f"step {step:5d}   loss {loss:.6f}")

print("\ntruth table after training:")
for xi, yi, pi in zip(X, y, p):
    print(f"  {int(xi[0])} XOR {int(xi[1])} -> {pi[0]:.3f}   (target {int(yi[0])})")

print("\nlearned hidden hyperplanes (w1*x1 + w2*x2 + b = 0):")
for i in range(2):
    print(f"  neuron {i}: {W1[i,0]:+.2f}*x1 {W1[i,1]:+.2f}*x2 {b1[i]:+.2f} = 0")

## `02_xor_pytorch.py`

```
Part 1 - XOR in PyTorch.

The exact same 2 -> 2 -> 1 network as 01_xor_numpy.py. Compare them line by
line: loss.backward() replaces the entire hand-written gradient block, and
optimizer.step() replaces the update lines. Nothing else changed.

Also saves a picture of the learned decision boundary to
../diagrams/xor-decision-boundary.png.

Run:  python 02_xor_pytorch.py
```

In [ ]:
import os

import torch
import torch.nn as nn

torch.manual_seed(42)

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

# The model: Linear stores W and b for us; Tanh/Sigmoid are the g's.
model = nn.Sequential(
    nn.Linear(2, 2),   # W1 (2,2), b1 (2,)   -- the hidden layer
    nn.Tanh(),
    nn.Linear(2, 1),   # W2 (1,2), b2 (1,)   -- the output neuron
    nn.Sigmoid(),
)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params}")  # 9, same as the numpy version

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

# THE training loop. You will write these five lines forever (section 1.8).
for step in range(10_000):
    optimizer.zero_grad()          # clear old gradients
    loss = criterion(model(X), y)  # forward
    loss.backward()                # backward: autograd runs backprop
    optimizer.step()               # theta <- theta - eta * grad
    if step % 1000 == 0:
        print(f"step {step:5d}   loss {loss.item():.6f}")

with torch.no_grad():
    p = model(X)
print("\ntruth table after training:")
for xi, yi, pi in zip(X, y, p):
    print(f"  {int(xi[0])} XOR {int(xi[1])} -> {pi.item():.3f}   (target {int(yi[0])})")

# The two hyperplanes the hidden layer chose (compare with exercise 1.1!)
W1, b1 = model[0].weight.data, model[0].bias.data
print("\nlearned hidden hyperplanes:")
for i in range(2):
    print(f"  neuron {i}: {W1[i,0]:+.2f}*x1 {W1[i,1]:+.2f}*x2 {b1[i]:+.2f} = 0")

In [ ]:
# ------------------------------------------------- decision boundary plot
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    raise SystemExit("matplotlib not installed; skipping the plot")

# Evaluate the network on a dense grid of the input square.
gx, gy = torch.meshgrid(torch.linspace(-0.5, 1.5, 200),
                        torch.linspace(-0.5, 1.5, 200), indexing="xy")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
with torch.no_grad():
    zz = model(grid).reshape(200, 200)

fig, ax = plt.subplots(figsize=(5, 4.5))
cs = ax.contourf(gx, gy, zz, levels=20, cmap="RdBu_r", vmin=0, vmax=1, alpha=0.85)
ax.contour(gx, gy, zz, levels=[0.5], colors="k", linewidths=1.5)
ax.scatter([0, 1], [0, 1], s=120, c="#1a3a8f", edgecolors="k", zorder=3, label="XOR = 0")
ax.scatter([0, 1], [1, 0], s=120, c="#d1495b", edgecolors="k", marker="s", zorder=3, label="XOR = 1")
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("XOR decision boundary learned by the 2-2-1 network")
ax.legend(loc="upper center")
fig.colorbar(cs, label="network output")
out = os.path.join(os.getcwd(), "..", "diagrams", "xor-decision-boundary.png")
fig.tight_layout()
fig.savefig(out, dpi=120)
plt.show()
print(f"\nsaved {os.path.normpath(out)}")

## `03_digits_mlp.py`

```
Part 1 - the digit recognizer, revisited with named parts.

A 64 -> 64 -> 10 MLP on scikit-learn's 8x8 digits dataset (a small cousin of
MNIST: 1797 images, ships with sklearn, no download). Every ingredient now
has a name from Part 1: cross-entropy (1.5), Adam (1.6), autograd (1.7),
and the five-line training loop (1.8).

Run:  python 03_digits_mlp.py        (~1 minute on CPU, ~97% test accuracy)
```

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)

In [ ]:
# ------------------------------------------------------------------- data
digits = load_digits()  # X: (1797, 64) pixel intensities 0..16, y: (1797,)
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.25, random_state=0)

# Normalizing inputs to ~[0,1] keeps pre-activations in the healthy range
# of the nonlinearity from step 0 (the same hygiene motif as LayerNorm).
X_train = torch.tensor(X_train, dtype=torch.float32) / 16.0
X_test = torch.tensor(X_test, dtype=torch.float32) / 16.0
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

In [ ]:
# ------------------------------------------------------------------ model
model = nn.Sequential(
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 10),   # 10 raw scores per image: the LOGITS.
)
# No softmax here: nn.CrossEntropyLoss applies log-softmax internally
# (numerically safer). The model outputs scores; the loss handles turning
# them into -log p_correct. GPT does exactly the same (Part 5).
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)
print(f"parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# --------------------------------------------------------------- training
batch_size = 64
for epoch in range(60):
    perm = torch.randperm(len(X_train))  # stochastic in SGD = shuffled batches
    for i in range(0, len(X_train), batch_size):
        idx = perm[i:i + batch_size]
        optimizer.zero_grad()
        loss = criterion(model(X_train[idx]), y_train[idx])
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        train_acc = (model(X_train).argmax(1) == y_train).float().mean()
        test_acc = (model(X_test).argmax(1) == y_test).float().mean()
    if epoch % 10 == 0 or epoch == 59:
        print(f"epoch {epoch:2d}  loss {loss.item():.4f}  "
              f"train acc {train_acc:.3f}  test acc {test_acc:.3f}")

In [ ]:
# -------------------------------------------- look at one prediction closely
with torch.no_grad():
    logits = model(X_test[:1])                    # (1, 10) raw scores
    probs = torch.softmax(logits, dim=1)[0]       # (10,)  sums to 1
print(f"\none test image, true digit = {y_test[0].item()}")
print("class probabilities from softmax(logits):")
for d, p in enumerate(probs):
    bar = "#" * int(p * 50)
    print(f"  {d}: {p:.3f} {bar}")
print(f"cross-entropy for this image = -log p_correct = "
      f"{-torch.log(probs[y_test[0]]).item():.4f}")